# Phase 2 — YoY-log demand regression

Notebook 03 ran OLS on first-differenced monthly data and produced an R² of 0.09 with no individually-significant regressors — the symptom of shared annual seasonality surviving $\Delta_1$ differencing. Here we rebuild on **year-over-year log change**, $\Delta_{12} \log x = \log x_t - \log x_{t-12}$, which removes the calendar cycle and keeps the slow macro signal intact.

Sequence:

1. Build YoY-log series for arrivals, the three real-FX rates, and the DE/GB Trends.
2. ADF and KPSS on the YoY series to cross-check stationarity.
3. OLS with Newey–West HAC (12 lags), Trends entering at YoY lag 1.
4. Re-fit with month-of-year fixed effects; compare $R^2$.
5. Economic reading of the real-EUR/TRY elasticity.
6. Robustness: drop the COVID window (2020-03 → 2022-02) and re-fit.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.stattools import durbin_watson
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_csv(ROOT / 'data' / 'processed' / 'master_monthly.csv',
                 index_col=0, parse_dates=True)
df.index.name = 'date'
print(df.shape)

(198, 29)


## 0. Defensive pulse-dummy derivation

The four 12-month pulse dummies (`war_pulse`, `mideast_pulse`, `covid_crash_pulse`, `covid_rebound_pulse`) are produced by `src/data_clean.add_shock_flags`. Older versions of `master_monthly.csv` do not have them, so the cell below derives them from the index if the columns are missing. Logic mirrors `data_clean.py` exactly.

In [2]:
idx = df.index
if 'war_pulse' not in df.columns:
    df['war_pulse']            = ((idx >= '2022-02-01') & (idx <= '2023-01-01')).astype(int)
    df['mideast_pulse']        = ((idx >= '2023-10-01') & (idx <= '2024-09-01')).astype(int)
    df['covid_crash_pulse']    = ((idx >= '2020-03-01') & (idx <= '2021-02-01')).astype(int)
    df['covid_rebound_pulse']  = ((idx >= '2021-03-01') & (idx <= '2022-02-01')).astype(int)
    print('pulse dummies derived from index (old master_monthly.csv)')
else:
    print('pulse dummies loaded from master_monthly.csv')

pulse dummies loaded from master_monthly.csv


## 1. YoY log-change transform

$yoy\_x_t = \log x_t - \log x_{t-12}$, expressed as a decimal (0.10 ≈ +10% YoY). Applied to arrivals, the three real-FX series, and the DE/GB Trends indices.

In [3]:
def yoy_log(s: pd.Series) -> pd.Series:
    return np.log(s) - np.log(s.shift(12))

def yoy_log_safe(s: pd.Series) -> pd.Series:
    # Trends indices can hit zero; log(0) is undefined -> drop those obs.
    s = s.replace(0, np.nan)
    return np.log(s) - np.log(s.shift(12))

df['yoy_arrivals']     = yoy_log(df['arrivals_total'])
df['yoy_real_EUR_TRY'] = yoy_log(df['real_EUR_TRY'])
df['yoy_real_GBP_TRY'] = yoy_log(df['real_GBP_TRY'])
df['yoy_real_USD_TRY'] = yoy_log(df['real_USD_TRY'])

df['yoy_trends_DE'] = yoy_log_safe(df['trends_DE_Türkei_Urlaub'])
df['yoy_trends_GB'] = yoy_log_safe(df['trends_GB_Turkey_holiday'])

yoy_cols = ['yoy_arrivals', 'yoy_real_EUR_TRY', 'yoy_real_GBP_TRY',
            'yoy_real_USD_TRY', 'yoy_trends_DE', 'yoy_trends_GB']
df[yoy_cols].describe().round(3)

,yoy_arrivals,yoy_real_EUR_TRY,yoy_real_GBP_TRY,yoy_real_USD_TRY,yoy_trends_DE,yoy_trends_GB
count,106.000,180.000,171.000,179.000,186.000,186.000
mean,0.079,0.039,0.047,0.053,0.011,-0.018
std,1.063,0.116,0.118,0.114,0.383,0.395
min,-5.057,-0.258,-0.237,-0.235,-1.504,-1.142
25%,0.048,-0.027,-0.021,-0.023,-0.162,-0.170
50%,0.127,0.043,0.052,0.043,0.000,-0.058
75%,0.276,0.119,0.124,0.149,0.151,0.105
max,3.790,0.387,0.392,0.412,1.407,1.598


## 2. ADF on the YoY series

Same ADF (constant, AIC-selected lag, 5% threshold) on `yoy_arrivals` and `yoy_real_EUR_TRY`.

In [4]:
def adf_row(name, s):
    s = s.dropna()
    stat, pval, used_lag, nobs, crit, _ = adfuller(s, regression='c', autolag='AIC')
    return {
        'series':     name,
        'n':          int(nobs),
        'used_lag':   int(used_lag),
        'adf_stat':   round(float(stat), 3),
        'p_value':    round(float(pval), 4),
        'crit_5pct':  round(float(crit['5%']), 3),
        'conclusion': 'stationary' if pval < 0.05 else 'non-stationary',
    }

adf_tab = pd.DataFrame([
    adf_row('yoy_arrivals',     df['yoy_arrivals']),
    adf_row('yoy_real_EUR_TRY', df['yoy_real_EUR_TRY']),
])
adf_tab

,series,n,used_lag,adf_stat,p_value,crit_5pct,conclusion
0,yoy_arrivals,103,2,-3.622,0.0054,-2.890,stationary
1,yoy_real_EUR_TRY,165,14,-2.269,0.1822,-2.879,non-stationary


## 2b. KPSS cross-check

ADF's null is a unit root, KPSS's null is (level) stationarity — so the two tests reverse the burden of proof. When they agree, the verdict is clean. When ADF says non-stationary and KPSS says stationary, the reading is that ADF has low power on samples this short (~100 obs on the YoY-differenced sample). When they both say non-stationary, inference on levels-like regressors becomes fragile and must be flagged in Limitations.

In [5]:
from statsmodels.tsa.stattools import kpss
import warnings
def kpss_row(name, s):
    s = s.dropna()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')  # kpss warns when p is outside table bounds
        stat, pval, lags, crit = kpss(s, regression='c', nlags='auto')
    return {'series': name, 'kpss_stat': round(float(stat), 3),
            'p_value': round(float(pval), 3),
            'conclusion': 'stationary' if pval > 0.05 else 'non-stationary'}

kpss_tab = pd.DataFrame([
    kpss_row('yoy_arrivals',     df['yoy_arrivals']),
    kpss_row('yoy_real_EUR_TRY', df['yoy_real_EUR_TRY']),
])
kpss_tab

,series,kpss_stat,p_value,conclusion
0,yoy_arrivals,0.100,0.1,stationary
1,yoy_real_EUR_TRY,0.186,0.1,stationary


**Interpretation.** The verdict cell below joins the ADF and KPSS columns. In this sample, KPSS fails to reject stationarity on both `yoy_arrivals` (KPSS stat 0.10, p ≈ 0.10) and `yoy_real_EUR_TRY` (KPSS stat 0.19, p ≈ 0.10). ADF rejects the unit root cleanly on `yoy_arrivals` (p = 0.005) but does not reject on `yoy_real_EUR_TRY` (p = 0.18). The two tests reverse the null, so the plain reading is: the YoY transform has removed the trend / unit-root component from both series, and ADF's non-rejection on the FX series is consistent with its well-known low power on small (~100 obs) samples. Both series are treated as stationary for the OLS below; the caveat is flagged in the README's Limitations bullet on stationarity.

In [6]:
verdict = adf_tab[['series','conclusion']].rename(columns={'conclusion':'adf'}).merge(
    kpss_tab[['series','conclusion']].rename(columns={'conclusion':'kpss'}),
    on='series')
verdict

,series,adf,kpss
0,yoy_arrivals,stationary,stationary
1,yoy_real_EUR_TRY,non-stationary,stationary


## 3. OLS — base specification

$$ yoy\_arrivals_t = \alpha
  + \beta_1\, yoy\_rEUR/TRY_t + \beta_2\, yoy\_rGBP/TRY_t
  + \beta_3\, yoy\_Trends^{DE}_{t-1} + \beta_4\, yoy\_Trends^{GB}_{t-1}
  + \gamma_1\,covid\_crash\_pulse + \gamma_2\,covid\_rebound\_pulse
  + \gamma_3\,war\_pulse + \gamma_4\,mideast\_pulse + \varepsilon_t $$

**Notes on this specification.**

- Shock controls are **12-month pulse dummies**, not step dummies. A permanent   level shift shows up in a $\Delta_{12}$-transformed dependent variable only   for the first 12 months (base-effect window), so YoY regressions need pulse,   not step, dummies to identify the shocks correctly.
- Trends enter as **YoY log change at lag 1**, not the raw 0–100 index.   The dependent variable is a deseasonalised YoY change; a raw seasonal 0–100   level regressor is on the wrong frequency and is mechanically biased toward a   near-zero coefficient. Placing both sides in YoY space removes that bias.   Zeros in the Trends index (log undefined) are dropped, which trims a small   number of observations from the estimation sample; the effective N is printed   below.
- Trends enter at **lag 1** deliberately. The YoY CCF in notebook 02 peaks at   lag 0 for GB with ρ = +0.462 (and at lag −6 for DE with ρ = +0.619 across   a broad monotonic profile), so using lag 0 would risk same-month   simultaneity between search intent and arrivals in a single observation.   Lag +1 on the GB YoY CCF is still above the 95% white-noise band   (ρ = +0.404, band ±0.191), so the trade-off is a possibly attenuated   coefficient in exchange for a cleaner leading-indicator interpretation.
- Standard errors are **Newey–West HAC with maxlags = 12**. The $\Delta_{12}$   transform on monthly data mechanically induces an MA(11) error structure   from overlapping differences, so the HAC bandwidth must be at least 11; we   use 12 to also absorb residual AR dynamics (DW is approximately 0.5 across   every spec).
- $\beta_1$ is an elasticity: a 1% real TRY depreciation against EUR is   associated with $\beta_1\%$ change in YoY arrivals.

In [7]:
model_df = df.assign(
    trends_DE_lag1 = df['yoy_trends_DE'].shift(1),
    trends_GB_lag1 = df['yoy_trends_GB'].shift(1),
    month          = df.index.month,
)

regressors = [
    'yoy_real_EUR_TRY', 'yoy_real_GBP_TRY',
    'trends_DE_lag1', 'trends_GB_lag1',
    'covid_crash_pulse', 'covid_rebound_pulse', 'war_pulse', 'mideast_pulse',
]
y_col = 'yoy_arrivals'

sample = model_df[[y_col, 'month'] + regressors].dropna()
print(f'Sample: {sample.index.min().date()} → {sample.index.max().date()}  (N={len(sample)})')

X_base = sm.add_constant(sample[regressors])
y_base = sample[y_col]
model_base = sm.OLS(y_base, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
print(model_base.summary())
print()
print(f'Durbin–Watson: {durbin_watson(model_base.resid):.3f}')

Sample: 2017-01-01 → 2025-03-01  (N=99)
                            OLS Regression Results                            
Dep. Variable:           yoy_arrivals   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.660
Method:                 Least Squares   F-statistic:                     27.11
Date:                Mon, 20 Jul 2026   Prob (F-statistic):           6.70e-21
Time:                        11:52:15   Log-Likelihood:                -91.812
No. Observations:                  99   AIC:                             201.6
Df Residuals:                      90   BIC:                             225.0
Df Model:                           8                                         
Covariance Type:                  HAC                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------

## 4. Add month fixed effects, compare R²

Month-of-year dummies absorb any residual calendar effect not removed by the YoY transform (e.g., shifting Eid/Easter, school-break timing). The comparison cell prints $R^2$ and the real-EUR/TRY coefficient under each spec.

In [8]:
formula = (
    'yoy_arrivals ~ yoy_real_EUR_TRY + yoy_real_GBP_TRY'
    ' + trends_DE_lag1 + trends_GB_lag1'
    ' + covid_crash_pulse + covid_rebound_pulse + war_pulse + mideast_pulse'
    ' + C(month)'
)
model_fe = smf.ols(formula, data=sample).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
print(model_fe.summary())
print()
print(f'Durbin–Watson: {durbin_watson(model_fe.resid):.3f}')

                            OLS Regression Results                            
Dep. Variable:           yoy_arrivals   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.613
Method:                 Least Squares   F-statistic:                     22.55
Date:                Mon, 20 Jul 2026   Prob (F-statistic):           2.46e-24
Time:                        11:52:15   Log-Likelihood:                -91.755
No. Observations:                  99   AIC:                             223.5
Df Residuals:                      79   BIC:                             275.4
Df Model:                          19                                         
Covariance Type:                  HAC                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.0423    

In [9]:
comp = pd.DataFrame({
    'no_FE': {
        'R2':         round(model_base.rsquared, 3),
        'Adj_R2':     round(model_base.rsquared_adj, 3),
        'beta_EUR':   round(model_base.params['yoy_real_EUR_TRY'], 3),
        'p_EUR':      round(model_base.pvalues['yoy_real_EUR_TRY'], 4),
        'DW':         round(durbin_watson(model_base.resid), 3),
    },
    'month_FE': {
        'R2':         round(model_fe.rsquared, 3),
        'Adj_R2':     round(model_fe.rsquared_adj, 3),
        'beta_EUR':   round(model_fe.params['yoy_real_EUR_TRY'], 3),
        'p_EUR':      round(model_fe.pvalues['yoy_real_EUR_TRY'], 4),
        'DW':         round(durbin_watson(model_fe.resid), 3),
    },
})
comp

,no_FE,month_FE
R2,0.6880,0.6880
Adj_R2,0.6600,0.6130
beta_EUR,1.6430,1.7250
p_EUR,0.1173,0.1641
DW,1.1620,1.1620


## 5. Economic interpretation — real-EUR/TRY elasticity

Cell below picks the **preferred spec** (higher adjusted-$R^2$) and translates its $\beta$ on `yoy_real_EUR_TRY` into the plain-English statement: *a 1% real TRY depreciation against EUR is associated with $\beta$% change in YoY arrivals*.

In [10]:
preferred_name, preferred = max(
    [('no-FE', model_base), ('month FE', model_fe)],
    key=lambda kv: kv[1].rsquared_adj,
)
b   = preferred.params['yoy_real_EUR_TRY']
p   = preferred.pvalues['yoy_real_EUR_TRY']
ci  = preferred.conf_int().loc['yoy_real_EUR_TRY']
alpha = 0.05

print(f'Preferred spec: {preferred_name}  (adj R² = {preferred.rsquared_adj:.3f})')
print(f'beta_EUR = {b:+.3f}   p = {p:.4f}   95% CI = [{ci[0]:+.3f}, {ci[1]:+.3f}]')
print()
verdict = 'significant at 5%' if p < alpha else 'NOT significant at 5%'
print(f'Result: real-EUR/TRY coefficient is {verdict}.')
print()
if p < alpha and b > 0:
    print('Economic reading:')
    print(f'  A 1% real TRY depreciation against EUR (yoy_real_EUR/TRY rises by 0.01)')
    print(f'  is associated with a {b:+.2f}% change in YoY arrivals.')
    print(f'  95% CI on that elasticity: [{ci[0]:+.2f}%, {ci[1]:+.2f}%].')
    print('  Sign matches the textbook substitution channel: cheaper TRY → more EU arrivals.')
elif p < alpha and b < 0:
    print(f'Sign is negative — a 1% real TRY depreciation is associated with {b:+.2f}% YoY arrivals.')
    print('  Counter to the textbook prior; likely picking up reverse-causal or omitted-variable dynamics.')
else:
    print(f'Point estimate: 1% real TRY depreciation ↔ {b:+.2f}% YoY arrivals.')
    print(f'  But the 95% CI [{ci[0]:+.2f}%, {ci[1]:+.2f}%] straddles zero — the data')
    print('  cannot rule out no FX effect at conventional levels. Interpret with caution.')

print()
print('Trends coefficients (YoY log at lag 1):')
for trend_name in ['trends_DE_lag1', 'trends_GB_lag1']:
    if trend_name in preferred.params.index:
        bt = preferred.params[trend_name]
        pt = preferred.pvalues[trend_name]
        cit = preferred.conf_int().loc[trend_name]
        sig = 'sig' if pt < alpha else 'ns'
        print(f'  {trend_name:18s}  beta={bt:+.3f}  p={pt:.4f}  95% CI=[{cit[0]:+.3f}, {cit[1]:+.3f}]  ({sig})')

print()
print('Other regressors at 5%:')
sig = preferred.pvalues[preferred.pvalues < alpha].drop(['yoy_real_EUR_TRY'], errors='ignore')
if sig.empty:
    print('  (no other regressor clears p < 0.05)')
else:
    for name in sig.index:
        if name.startswith('C(month)'): continue
        print(f'  {name:25s}  beta={preferred.params[name]:+.3e}  p={sig[name]:.4f}')

Preferred spec: no-FE  (adj R² = 0.660)
beta_EUR = +1.643   p = 0.1173   95% CI = [-0.413, +3.699]

Result: real-EUR/TRY coefficient is NOT significant at 5%.

Point estimate: 1% real TRY depreciation ↔ +1.64% YoY arrivals.
  But the 95% CI [-0.41%, +3.70%] straddles zero — the data
  cannot rule out no FX effect at conventional levels. Interpret with caution.

Trends coefficients (YoY log at lag 1):
  trends_DE_lag1      beta=-0.483  p=0.0543  95% CI=[-0.975, +0.009]  (ns)
  trends_GB_lag1      beta=+0.858  p=0.0280  95% CI=[+0.092, +1.623]  (sig)

Other regressors at 5%:


  trends_GB_lag1             beta=+8.576e-01  p=0.0280
  covid_crash_pulse          beta=-1.891e+00  p=0.0000
  covid_rebound_pulse        beta=+1.670e+00  p=0.0005


## 6. Robustness: excluding the COVID window

The YoY arrivals series shows a very large positive swing over 2020–2022 as the pandemic collapse rolls off the 12-month base. The two COVID pulse dummies control for it in the full sample, but a second look — dropping the whole window (2020-03 → 2022-02) and re-fitting — asks whether the elasticity is *driven by* those months or *survives* them.

In this subsample the COVID pulses are all zero, so we drop them from the regressor list (otherwise the design matrix is singular). `war_pulse` and `mideast_pulse` are kept — both fall partly outside the excluded window.

In [11]:
covid_mask = (sample.index >= '2020-03-01') & (sample.index <= '2022-02-01')
sample_exc = sample.loc[~covid_mask].copy()
print(f'ex-COVID sample: {sample_exc.index.min().date()} → {sample_exc.index.max().date()}  (N={len(sample_exc)})')

regressors_exc = [
    'yoy_real_EUR_TRY', 'yoy_real_GBP_TRY',
    'trends_DE_lag1', 'trends_GB_lag1',
    'war_pulse', 'mideast_pulse',
]
X_exc = sm.add_constant(sample_exc[regressors_exc])
y_exc = sample_exc[y_col]
model_exc = sm.OLS(y_exc, X_exc).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
print(model_exc.summary())
print()
print(f'Durbin–Watson: {durbin_watson(model_exc.resid):.3f}')

ex-COVID sample: 2017-01-01 → 2025-03-01  (N=75)
                            OLS Regression Results                            
Dep. Variable:           yoy_arrivals   R-squared:                       0.680
Model:                            OLS   Adj. R-squared:                  0.652
Method:                 Least Squares   F-statistic:                     32.34
Date:                Mon, 20 Jul 2026   Prob (F-statistic):           4.23e-18
Time:                        11:52:15   Log-Likelihood:                 48.649
No. Observations:                  75   AIC:                            -83.30
Df Residuals:                      68   BIC:                            -67.08
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

In [12]:
def row(name, m, fx='yoy_real_EUR_TRY'):
    ci = m.conf_int().loc[fx]
    return {
        'spec':      name,
        'N':         int(m.nobs),
        f'beta_{fx}': round(float(m.params[fx]), 3),
        f'p_{fx}':    round(float(m.pvalues[fx]), 4),
        'ci_lo':     round(float(ci[0]), 3),
        'ci_hi':     round(float(ci[1]), 3),
        'R2':        round(float(m.rsquared), 3),
    }

cmp_covid = pd.DataFrame([row('full', model_base), row('ex-COVID', model_exc)]).set_index('spec')
cmp_covid

,N,beta_yoy_real_EUR_TRY,p_yoy_real_EUR_TRY,ci_lo,ci_hi,R2
spec,,,,,,
full,99,1.643,0.1173,-0.413,3.699,0.688
ex-COVID,75,-0.516,0.2470,-1.388,0.357,0.680


**Reading the comparison.** The verdict cell below prints whether the real-EUR/TRY point estimate survives dropping the COVID window and whether its p-value stays below 5%. If both hold, the headline elasticity is not a pandemic artefact. If the point estimate collapses or the p-value blows up, the finding is COVID-window dependent and the README/article must report it as such.

In [13]:
b_full,  p_full  = model_base.params['yoy_real_EUR_TRY'], model_base.pvalues['yoy_real_EUR_TRY']
b_exc,   p_exc   = model_exc.params['yoy_real_EUR_TRY'],  model_exc.pvalues['yoy_real_EUR_TRY']
print(f'full     : beta={b_full:+.3f}  p={p_full:.4f}')
print(f'ex-COVID : beta={b_exc:+.3f}  p={p_exc:.4f}')
print()
if p_exc < 0.05 and b_exc * b_full > 0:
    print('Verdict: the EUR/TRY elasticity survives the ex-COVID cut with the same sign at 5%.')
    print('The full-sample finding is not driven by pandemic swings.')
elif b_exc * b_full > 0:
    print('Verdict: same sign, but the ex-COVID p-value is above 5%.')
    print('The full-sample significance is at least partly COVID-window dependent.')
else:
    print('Verdict: the sign of the EUR/TRY elasticity flips (or its magnitude collapses)')
    print('when the COVID window is dropped. The full-sample headline is COVID-window dependent')
    print('and must be reported as such.')

full     : beta=+1.643  p=0.1173
ex-COVID : beta=-0.516  p=0.2470

Verdict: the sign of the EUR/TRY elasticity flips (or its magnitude collapses)
when the COVID window is dropped. The full-sample headline is COVID-window dependent
and must be reported as such.
